In [1]:
pip install flask joblib requests scikit-learn numpy pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import joblib
import requests
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

### Task 1: Train & Serialize

1. Load the Iris dataset (`load_iris()`). Split into 80/20 train/test (`random_state=42`).
2. Train a `RandomForestClassifier(n_estimators=100, random_state=42)` on the training set.
3. Evaluate on the test set — print the **accuracy** and the full **classification report**.
4. Save the trained model to a file called `model.joblib` using `joblib.dump()`.
5. In a new cell, load the model back with `joblib.load()` and predict on the same test set. Verify that the predictions are **identical** to the original (use `np.array_equal`).
6. Also save the **target names** (species labels) to `target_names.joblib` — the API will need them to return human-readable class names.


In [3]:
iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train samples: {len(X_train)}")
print(f"Test samples:  {len(X_test)}")


Train samples: 120
Test samples:  30


In [4]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print("Training complete.")

Training complete.


In [5]:
y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print()
print(classification_report(y_test, y_pred, target_names=iris.target_names))

Accuracy: 1.0000

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



In [7]:
joblib.dump(model, "model.joblib")
joblib.dump(iris.target_names, "target_names.joblib")

print("Saved: model.joblib")
print("Saved: target_names.joblib")

Saved: model.joblib
Saved: target_names.joblib


In [8]:
model_reloaded = joblib.load("model.joblib")
y_pred_reloaded = model_reloaded.predict(X_test)

print("Predictions identical:", np.array_equal(y_pred, y_pred_reloaded))

Predictions identical: True


### Task 2: Build a Flask API

Create a file called **`app.py`** with the following structure:

1. **`/health` endpoint** (GET): Returns `{"status": "healthy"}` with a 200 status code. This is used by load balancers and monitoring tools to check if the service is alive.

2. **`/predict` endpoint** (POST): Accepts a JSON body with a `features` key containing a list of 4 numeric values (sepal length, sepal width, petal length, petal width).
   - Load the model and target names from the `.joblib` files.
   - Validate the input:
     - Check that the `features` key exists.
     - Check that it contains exactly 4 values.
     - Check that all values are numeric.
     - Return a 400 error with a descriptive message if validation fails.
   - Return a JSON response with `predicted_class` (string name) and `probabilities` (dict mapping class names to probabilities).

3. **`/predict_batch` endpoint** (POST): Accepts a JSON body with a `samples` key containing a list of feature arrays. Returns predictions for all samples in a single response.

4. Run the app with `app.run(debug=True, port=5000)`.

Here's a skeleton to get you started:

```python
from flask import Flask, request, jsonify
import joblib
import numpy as np

app = Flask(__name__)

model = joblib.load("model.joblib")
target_names = joblib.load("target_names.joblib")

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "healthy"})

# Implement /predict and /predict_batch below

if __name__ == "__main__":
    app.run(debug=True, port=5000)
```

In [9]:
print(open("app.py").read())

from flask import Flask, request, jsonify
import joblib
import numpy as np

app = Flask(__name__)

model = joblib.load("model.joblib")
target_names = joblib.load("target_names.joblib")


@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "healthy"}), 200


@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json()

    if data is None or "features" not in data:
        return jsonify({"error": "Missing 'features' key in request body"}), 400

    features = data["features"]

    if len(features) != 4:
        return jsonify({
            "error": f"Expected exactly 4 feature values, got {len(features)}"
        }), 400

    for i, v in enumerate(features):
        if not isinstance(v, (int, float)):
            return jsonify({
                "error": f"All values must be numeric. Got non-numeric at index {i}: {v!r}"
            }), 400

    X = np.array(features).reshape(1, -1)
    pred_idx = model.predict(X)[0]
    proba = mo

### Task 3: Test the API

**Before starting this task, open a terminal and run `python app.py` to start the Flask server.**

Back in your notebook:

1. **Health check**: Send a GET request to `http://localhost:5000/health` and verify the response.
2. **Single prediction**: Send a POST request to `/predict` with a valid Iris sample (e.g., `[5.1, 3.5, 1.4, 0.2]`). Print the predicted class and probabilities.
3. **Error handling**: Send at least three malformed requests and verify that the API returns appropriate 400 errors:
   - Missing `features` key.
   - Wrong number of features (e.g., 3 instead of 4).
   - Non-numeric values (e.g., `["a", "b", "c", "d"]`).
4. **Batch prediction**: Send a POST request to `/predict_batch` with 5 samples from the Iris test set. Verify that all predictions match what the model produces locally.
5. Document all test results in the notebook with clear output and markdown explanations.


In [10]:
import urllib.request, json

def api(method, path, data=None):
    url = f"http://127.0.0.1:5000{path}"
    body = json.dumps(data).encode() if data else None
    req = urllib.request.Request(
        url, data=body, method=method,
        headers={"Content-Type": "application/json"} if body else {}
    )
    try:
        resp = urllib.request.urlopen(req)
        return resp.status, json.loads(resp.read())
    except urllib.error.HTTPError as e:
        return e.code, json.loads(e.read())

In [11]:
status, body = api("GET", "/health")
print(f"Status : {status}")
print(f"Body   : {body}")
assert status == 200 and body["status"] == "healthy"
print("✓ Health check passed")

Status : 200
Body   : {'status': 'healthy'}
✓ Health check passed


In [12]:
status, body = api("POST", "/predict", {"features": [5.1, 3.5, 1.4, 0.2]})
print(f"Status          : {status}")
print(f"Predicted class : {body['predicted_class']}")
print(f"Probabilities   :")
for cls, prob in body["probabilities"].items():
    print(f"  {cls:>12} : {prob}")
assert status == 200
print("✓ Single prediction passed")

Status          : 200
Predicted class : setosa
Probabilities   :
        setosa : 1.0
    versicolor : 0.0
     virginica : 0.0
✓ Single prediction passed


In [13]:
status, body = api("POST", "/predict", {"data": [5.1, 3.5, 1.4, 0.2]})
print(f"Status  : {status}")
print(f"Message : {body['error']}")
assert status == 400
print("✓ Correct 400 returned")

Status  : 400
Message : Missing 'features' key in request body
✓ Correct 400 returned


In [14]:
status, body = api("POST", "/predict", {"features": [5.1, 3.5, 1.4]})
print(f"Status  : {status}")
print(f"Message : {body['error']}")
assert status == 400
print("✓ Correct 400 returned")

Status  : 400
Message : Expected exactly 4 feature values, got 3
✓ Correct 400 returned


In [15]:
status, body = api("POST", "/predict", {"features": ["a", "b", "c", "d"]})
print(f"Status  : {status}")
print(f"Message : {body['error']}")
assert status == 400
print("✓ Correct 400 returned")

Status  : 400
Message : All values must be numeric. Got non-numeric at index 0: 'a'
✓ Correct 400 returned


In [16]:
samples_5 = X_test[:5].tolist()
local_preds = model.predict(X_test[:5])
local_classes = [iris.target_names[i] for i in local_preds]

status, body = api("POST", "/predict_batch", {"samples": samples_5})
api_classes = [p["predicted_class"] for p in body["predictions"]]

print(f"Status            : {status}")
print(f"API predictions   : {api_classes}")
print(f"Local predictions : {list(local_classes)}")
print(f"All match         : {api_classes == list(local_classes)}")
assert status == 200 and api_classes == list(local_classes)
print("✓ Batch prediction matches local model")

Status            : 200
API predictions   : ['versicolor', 'setosa', 'virginica', 'versicolor', 'versicolor']
Local predictions : [np.str_('versicolor'), np.str_('setosa'), np.str_('virginica'), np.str_('versicolor'), np.str_('versicolor')]
All match         : True
✓ Batch prediction matches local model


### Task 4: Documentation & Reflection

1. Create a `README.md` file (separate from this lab README) inside a `deployment/` folder that documents your deployed model:
   - **What the model does** (one paragraph).
   - **How to run** (step-by-step commands).
   - **API specification** (endpoints, methods, request/response formats).
   - **Example requests and responses** (curl or Python `requests` examples).

2. In your notebook, write a reflection addressing:
   - What would need to change for a **production deployment**? (Think about: WSGI servers, containerization, environment variables, HTTPS.)
   - How would you handle **model versioning**? (What if you retrain and the new model is worse?)
   - What **monitoring** would you add? (Prediction latency, input drift, error rates.)
   - How would the architecture change if the model needed to handle **1,000 requests per second**?


#### What would need to change for a production deployment?

Flask's built-in server is single-threaded and explicitly warns you not to use it in production. The first change is replacing it with a proper WSGI server — **Gunicorn**. For async workloads, Gunicorn can be combined with Uvicorn workers to support an ASGI-style execution model when needed. In front of the application, **Nginx** acts as a reverse proxy: it handles TLS termination, connection management, and static file serving.

Everything should run inside a Docker container with pinned dependency versions to ensure consistency across environments. The image is built in CI and deployed identically everywhere.

Configuration such as ports, worker counts, and secrets must not be hardcoded. Use environment variables (`os.environ`) or a secrets manager like AWS Secrets Manager or HashiCorp Vault.

HTTPS must be enforced. TLS termination is handled at the load balancer or Nginx layer, with certificates managed and rotated automatically (e.g., via Let's Encrypt).

#### How would you handle model versioning?

Never overwrite a model that is currently serving traffic. Each training run should save its artifact to a versioned path such as `models/v{timestamp}/model.joblib`, along with evaluation metrics, training configuration, and dataset version or hash.

A new model is only promoted if it outperforms the current one on a fixed validation set. This process can be automated in CI/CD pipelines. The previous model must remain available for instant rollback by simply switching configuration.

For safer deployment, use canary or A/B testing: route a small percentage of traffic to the new model, monitor real-world performance, and then either promote it fully or roll it back.

#### What monitoring would you add?

Monitoring should include latency, error rates, and data drift. Track p50, p95, and p99 latency and alert if thresholds are exceeded. Monitor 4xx and 5xx error rates separately to distinguish between client and server issues.

Log feature inputs and prediction outputs, and compare them with training data distributions using statistical tests (e.g., KS test or KL divergence) to detect input and prediction drift.

A critical addition is a feedback loop: when real-world labels become available, measure actual model accuracy in production to detect silent failures that basic metrics might miss.

#### How would the architecture change at 1,000 requests per second?

At 1k requests per second, horizontal scaling becomes essential. Deploy multiple Gunicorn instances behind a load balancer and use Kubernetes with Horizontal Pod Autoscaling. Use the `--preload` option so the model is loaded once per pod and shared across workers.

Caching is generally ineffective for ML inputs because identical feature vectors are rare, so batching is more valuable. Dynamic batching improves throughput by processing multiple requests together.

For long-running or large requests, use asynchronous processing with a task queue like Celery and Redis. Return a job ID and allow clients to poll for results.

For larger or more complex models, use dedicated model serving systems such as Triton Inference Server for better resource utilization and concurrency.

Improve observability with structured JSON logging, log sampling, and distributed tracing (OpenTelemetry + Jaeger). Finally, add rate limiting, authentication (API keys or JWT), and request size limits to ensure security and system stability.